## Data Ingestion

#### Langchain Documentaion Structure 


    Reads files from a directory and ingests them into LangChain documents.

In [ ]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredFileLoader

#Data to be ingetsted from directory and converted to Langchain documents

def ingest_data_from_directory(directory_path: str):
    """
    Reads files from a directory and ingests them into LangChain documents.
    Supports TXT, PDF, and other unstructured formats.
    """
    documents = []

    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)

        if filename.endswith(".txt"):
            loader = TextLoader(file_path)
        elif filename.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
        else:
            # fallback for other formats
            loader = UnstructuredFileLoader(file_path)

        docs = loader.load()
        documents.extend(docs)

    return documents



directory = "../data"
docs = ingest_data_from_directory(directory)
print(f"Ingested {len(docs)} chunks from {directory}")

In [ ]:
docs

In [ ]:


from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    split_docs = text_splitter.split_documents(documents)
    return split_docs

In [ ]:

split_docs = split_documents(docs)
split_docs

In [ ]:
len(docs), len(split_docs)

In [ ]:



from typing import List
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        self.model = None
        self.__load_model()
        
    def __load_model(self):
        self.model = SentenceTransformer(self.model_name)
        
    def generate_embedding(self, text:List[str]):
        print("Generating embeddings... for length:", len(text))
        embeddings =  self.model.encode(text,show_progress_bar=True)
        return embeddings

#initialize embedding manager and generate embeddings
embedding_manager = EmbeddingManager()
embedding_manager
       

## Vector Store

In [ ]:


from typing import Any, List
import numpy as np
from sqlalchemy import desc


class VectorStoreManager:
    def __init__(self, collectiona_name="resumes_document",persist_directory="../data/vector_store"):
        import chromadb
        from chromadb.config import Settings

        self.collectiona_name = collectiona_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.collection = self.__initialize_store()
        
    def __initialize_store(self):
        import chromadb
        from chromadb.config import Settings

        self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )
        self.collection = self.client.get_or_create_collection(
            name=self.collectiona_name,
            metadata={"description": "Resume Embeddings for RAG"},
            )
        return self.collection
        
    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        #add documents to vector store
        ids = []
        metadatas=[]
        texts = []
        embeddings_list = []

        if len(documents) != len(embeddings):
            raise ValueError("Length of documents and embeddings must be equal.")

        for i, doc in enumerate(documents):
            ids.append(f"doc_{i}")
            texts.append(doc.page_content)
            embeddings_list.append(embeddings[i])

        self.collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings_list
        )
        
        
        print(f"Added {len(documents)} documents to the vector store.")
        

In [ ]:

vecdotr_store_manager = VectorStoreManager()

In [ ]:
vecdotr_store_manager

In [ ]:
#generate embeddings for split documents
texts = [doc.page_content for doc in split_docs]
texts

In [ ]:
#generate embeddings for split documents
embeddings = embedding_manager.generate_embedding(texts)

#add to vector store
vecdotr_store_manager.add_documents(split_docs, embeddings)


## Rag Retriever

In [ ]:



class Rag_Retriever:
    def __init__(self, vector_store_manager, embedding_manager):
        self.vector_store_manager = vector_store_manager
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5,score_threshold: float = 0.0):
        print(f"Retrieving top {top_k} documents for query: {query}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        
        # Perform similarity search in the vector store
        results = self.vector_store_manager.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        
        retrieved_docs = []
        
        if(results['documents'] and results['documents'][0]):
            documents = results['documents'][0]
            metadatas = results['metadatas'][0]
            distances = results['distances'][0]
            ids = results['ids'][0]
        
            for i,(doc_id, doc, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                print(f"Document ID: {doc_id}")
                print(f"Distance: {distance}")
                print(f"Metadata: {metadata}")
                print(f"Content: {doc[:200]}...")  # Print first 200 characters
                print("-" * 50)
                
                similarity_score = 1 - distance  # Assuming distance is in [0, 2] for cosine similarity
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": doc,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

        return retrieved_docs  # Return the list of retrieved documents
    
Rag_Retriever= Rag_Retriever(vecdotr_store_manager, embedding_manager)
Rag_Retriever.retrieve("Skills", top_k=3, score_threshold=0.1)

In [ ]:
Rag_Retriever.retrieve("Skills", top_k=3, score_threshold=0.1)

## Integrate vecor db context pipleine with vector db